# Hybrid Video Search with CLIP + DINOv3 + LanceDB

This notebook demonstrates how to build a powerful multimodal video search system that combines:
- **CLIP**: For semantic text-to-image search
- **DINOv3**: For fine-grained visual features
- **LanceDB**: For efficient vector storage and retrieval

## Setup

In [ ]:
# Install dependencies
!pip install -r requirements.txt

In [ ]:
from video_search import HybridVideoSearch
import matplotlib.pyplot as plt
from PIL import Image
import os

## 1. Initialize the Search System

In [ ]:
# Initialize hybrid video search
search = HybridVideoSearch(
    db_path="./video_search.db",
    clip_model="ViT-B-32",
    device="cuda",  # Use "cpu" if no GPU available
    use_dino=True,  # Enable DINOv3 for better visual precision
    batch_size=32
)

print("✓ Search system initialized!")

## 2. Index a Video

Extract frames from a video and compute embeddings.

In [ ]:
# Path to your video file
video_path = "path/to/your/video.mp4"
video_id = "demo_video_001"

# Index the video
num_frames = search.index_video(
    video_path=video_path,
    video_id=video_id,
    fps=2,  # Extract 2 frames per second
    max_frames=None,  # Process all frames (or set a limit)
    cleanup_frames=False  # Keep frames for visualization
)

print(f"✓ Indexed {num_frames} frames from {video_id}")

## 3. Text Search

Search for frames using natural language descriptions.

In [ ]:
# Search with text query
query = "American flag in the background"

results = search.search_text(query, limit=10)

print(f"Found {len(results)} results for: '{query}'\n")

for i, result in enumerate(results[:5], 1):
    print(f"{i}. Video: {result['video_id']}")
    print(f"   Frame: {result['frame_id']} at {result['timestamp']:.2f}s")
    print(f"   Score: {result['score']:.3f}")
    print(f"   Path: {result['frame_path']}")
    print()

### Visualize Top Results

In [ ]:
# Display top 6 results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, result in enumerate(results[:6]):
    if os.path.exists(result['frame_path']):
        img = Image.open(result['frame_path'])
        axes[i].imshow(img)
        axes[i].set_title(f"Frame {result['frame_id']} - {result['timestamp']:.1f}s\nScore: {result['score']:.3f}")
        axes[i].axis('off')

plt.tight_layout()
plt.show()

## 4. Image Similarity Search

Find frames visually similar to a query image.

In [ ]:
# Query image path
query_image = "path/to/query_image.jpg"

# Search for similar frames
results = search.search_image(
    image_path=query_image,
    limit=10,
    clip_weight=0.5,  # Semantic similarity
    dino_weight=0.5   # Visual similarity
)

print(f"Found {len(results)} similar frames")

In [ ]:
# Display query image and top results
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Show query image
if os.path.exists(query_image):
    query_img = Image.open(query_image)
    axes[0].imshow(query_img)
    axes[0].set_title("Query Image", fontsize=14, fontweight='bold')
    axes[0].axis('off')

# Show results
for i, result in enumerate(results[:7], 1):
    if os.path.exists(result['frame_path']):
        img = Image.open(result['frame_path'])
        axes[i].imshow(img)
        axes[i].set_title(f"Result {i}\nScore: {result['score']:.3f}")
        axes[i].axis('off')

plt.tight_layout()
plt.show()

## 5. Hybrid Search (Text + Image)

Combine text and image queries for more precise results.

In [ ]:
# Hybrid search
results = search.hybrid_search(
    text="person wearing red shirt",
    image=query_image,
    text_weight=0.6,
    image_weight=0.4,
    limit=10
)

print(f"Hybrid search found {len(results)} results")

for i, result in enumerate(results[:5], 1):
    print(f"{i}. Frame {result['frame_id']} at {result['timestamp']:.2f}s - Score: {result['score']:.3f}")

## 6. Object Localization

Find and localize specific objects within frames using text descriptions.

In [ ]:
# Select a frame to analyze
frame_path = results[0]['frame_path'] if results else "path/to/frame.jpg"

# Detect objects
detections = search.locate_object(
    frame_path=frame_path,
    query="American flag",
    threshold=0.25
)

print(f"Found {len(detections)} detection(s)")

for i, det in enumerate(detections, 1):
    x, y, w, h = det['bbox']
    print(f"{i}. BBox: ({x}, {y}, {w}, {h}), Score: {det['score']:.3f}")

In [ ]:
# Visualize detections
if detections:
    viz_img = search.visualize_detections(
        frame_path=frame_path,
        detections=detections,
        output_path="detection_result.jpg"
    )
    
    # Display side-by-side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Original
    original = Image.open(frame_path)
    ax1.imshow(original)
    ax1.set_title("Original Frame", fontsize=14)
    ax1.axis('off')
    
    # With detections
    ax2.imshow(viz_img)
    ax2.set_title(f"Detections ({len(detections)} found)", fontsize=14)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No detections found")

## 7. Multi-Query Search

Test multiple queries to explore the video content.

In [ ]:
queries = [
    "person's face",
    "outdoor scene",
    "American flag",
    "building",
    "car",
    "crowd of people"
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, query in enumerate(queries):
    results = search.search_text(query, limit=1)
    
    if results and os.path.exists(results[0]['frame_path']):
        img = Image.open(results[0]['frame_path'])
        axes[i].imshow(img)
        axes[i].set_title(f"Query: '{query}'\nScore: {results[0]['score']:.3f}", fontsize=12)
    else:
        axes[i].text(0.5, 0.5, f"No results for\n'{query}'", 
                    ha='center', va='center', fontsize=12)
    
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 8. Database Statistics

In [ ]:
# Get database stats
stats = search.get_stats()

print("Database Statistics:")
print("=" * 50)
print(f"Database Path: {search.db_path}")
print(f"Tables: {stats['tables']}")
print(f"Total Frames Indexed: {stats['total_frames']}")
print(f"Number of Videos: {len(stats['videos'])}")

if stats['videos']:
    print(f"\nVideo IDs:")
    for vid in stats['videos']:
        print(f"  - {vid}")

## 9. Advanced: Temporal Object Tracking

Track an object across multiple frames in the video timeline.

In [ ]:
# Search for frames containing target object
target_query = "person wearing red"
results = search.search_text(target_query, limit=20)

print(f"Found {len(results)} frames with '{target_query}'")

# Track object across frames
tracking_data = []

for result in results[:10]:  # Process top 10
    detections = search.locate_object(
        frame_path=result['frame_path'],
        query=target_query,
        threshold=0.25
    )
    
    if detections:
        tracking_data.append({
            'timestamp': result['timestamp'],
            'frame_id': result['frame_id'],
            'num_detections': len(detections),
            'best_score': max(d['score'] for d in detections),
            'frame_path': result['frame_path']
        })

print(f"\nTracking Timeline ({len(tracking_data)} frames):")
for track in tracking_data:
    print(f"  Time: {track['timestamp']:6.2f}s | Frame: {track['frame_id']:4d} | "
          f"Detections: {track['num_detections']} | Best Score: {track['best_score']:.3f}")

In [ ]:
# Plot tracking timeline
if tracking_data:
    timestamps = [t['timestamp'] for t in tracking_data]
    scores = [t['best_score'] for t in tracking_data]
    num_dets = [t['num_detections'] for t in tracking_data]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    
    # Plot scores over time
    ax1.plot(timestamps, scores, 'b-o', linewidth=2, markersize=8)
    ax1.set_xlabel('Time (seconds)', fontsize=12)
    ax1.set_ylabel('Detection Score', fontsize=12)
    ax1.set_title(f'Object Detection Scores Over Time: "{target_query}"', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot number of detections
    ax2.bar(timestamps, num_dets, width=0.5, color='green', alpha=0.7)
    ax2.set_xlabel('Time (seconds)', fontsize=12)
    ax2.set_ylabel('Number of Detections', fontsize=12)
    ax2.set_title('Number of Detections Per Frame', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## 10. Export Results

Save search results for further analysis.

In [ ]:
import json

# Export search results
query = "American flag"
results = search.search_text(query, limit=20)

# Save to JSON
output_file = "search_results.json"
with open(output_file, 'w') as f:
    json.dump({
        'query': query,
        'num_results': len(results),
        'results': results
    }, f, indent=2)

print(f"✓ Saved {len(results)} results to {output_file}")

## Conclusion

This notebook demonstrated:

1. ✅ Indexing videos with CLIP + DINOv3 embeddings
2. ✅ Text-based semantic search
3. ✅ Image similarity search
4. ✅ Hybrid text+image search
5. ✅ Object localization and detection
6. ✅ Temporal object tracking

### Next Steps:

- Index multiple videos for larger-scale search
- Experiment with different CLIP models (ViT-L-14 for better accuracy)
- Adjust thresholds and weights for your specific use case
- Integrate with web interface or API
- Add custom post-processing for domain-specific applications